# Type System & Protocols — Expert Interview Guide

Covers: type hints, `Protocol`, `TypedDict`, `Annotated`, `Generic`, `@overload`, Pydantic v2.

> **Key insight:** Python's type system is **structural** (Protocol) not just nominal (inheritance). If it has the right methods, it satisfies the protocol.

## 1. Type Hints Basics

Type hints are **optional metadata** — not enforced at runtime. Use `mypy` for static checking.

In [ ]:
list[int]          # Python 3.9+
dict[str, int]
tuple[int, ...]    # variable-length homogenous
str | None         # Python 3.10+ (same as Optional[str])
int | str          # same as Union[int, str]

In [ ]:
from typing import Optional, Union

def greet(name: str, times: int = 1) -> str:
    return (f'Hello, {name}! ' * times).strip()

print(greet('Alice', 2))

def find_user(uid: int) -> Optional[str]:
    users = {1: 'Alice', 2: 'Bob'}
    return users.get(uid)

print(find_user(1), find_user(99))

def process(value: Union[int, str]) -> str:
    if isinstance(value, int): return f'Number: {value*2}'
    return f'String: {value.upper()}'

print(process(5), process('hello'))

# Type hints NOT enforced at runtime!
def strict(x: int) -> int: return x * 2
print(strict('ha'))  # 'haha' -- no runtime error!

> **Interview Insight:** Type hints are completely ignored by the Python runtime. `def f(x: int): pass; f('str')` works fine. Use `mypy --strict` for static checking, or `beartype` for runtime enforcement.

## 2. Protocols & Structural Subtyping

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...

# These satisfy Drawable WITHOUT inheriting
class Circle:
    def draw(self) -> str: return f'Circle'

class Square:
    def draw(self) -> str: return 'Square()'

def render_all(shapes: list[Drawable]) -> None:
    for s in shapes: print(s.draw())

render_all([Circle(), Square()])

# runtime_checkable enables isinstance
print(f'Circle is Drawable: {isinstance(Circle(), Drawable)}')

> **Interview Insight:** `@runtime_checkable` checks method **existence only**, not signatures. `isinstance(obj, MyProto)` won't verify argument types. For full structural checking, rely on `mypy` static analysis.

## 3. Generics & TypeVar

In [ ]:
from typing import TypeVar, Generic, Sequence

T = TypeVar('T')

def first(seq: Sequence[T]) -> T: return seq[0]
print(first([1,2,3]))    # int
print(first(['a','b']))  # str

class Stack(Generic[T]):
    def __init__(self) -> None: self._items: list[T] = []
    def push(self, item: T) -> None: self._items.append(item)
    def pop(self) -> T:
        if not self._items: raise IndexError('empty')
        return self._items.pop()

s: Stack[int] = Stack()
s.push(1); s.push(2); s.push(3)
print(s.pop())

> **Interview Insight:** Generic types are erased at runtime — `Stack[int]` and `Stack[str]` are the same class. Use `get_args(Stack[int])` from `typing` if you need to inspect type params at runtime.

## 4. TypedDict & Annotated

In [ ]:
from typing import TypedDict, NotRequired, Annotated, get_type_hints, get_args

class UserProfile(TypedDict):
    id: int
    name: str
    email: str

class Employee(TypedDict):
    id: int
    name: str
    department: NotRequired[str]  # optional key

user: UserProfile = {'id': 1, 'name': 'Alice', 'email': 'a@b.com'}
emp: Employee = {'id': 2, 'name': 'Bob'}  # department optional
print(user, emp)

# Annotated -- attach metadata to type hints
class Gt:
    def __init__(self, v): self.v = v

Age = Annotated[int, Gt(0)]

def create_user(age: Age, name: str) -> dict:
    return {'age': age, 'name': name}

hints = get_type_hints(create_user, include_extras=True)
base, *validators = get_args(hints['age'])
print(f'Base type: {base}')
print(f'Validators: {[type(v).__name__ for v in validators]}')

> **Interview Insight:** `TypedDict` is a type-checker fiction — at runtime it's just `dict`. FastAPI and Pydantic use `Annotated` extensively: `Annotated[int, Query(gt=0)]` tells FastAPI to validate query params.

## 5. NewType & TYPE_CHECKING

In [ ]:
from typing import NewType, TypeAlias, TYPE_CHECKING, cast, Any

# TypeAlias -- same type, just a name
Vector: TypeAlias = list[float]

# NewType -- distinct type for type checker (no-op at runtime)
UserId = NewType('UserId', int)
ProductId = NewType('ProductId', int)

def get_user(uid: UserId) -> str: return f'User {uid}'
def get_product(pid: ProductId) -> str: return f'Product {pid}'

uid = UserId(42); pid = ProductId(42)
print(get_user(uid))      # OK
print(get_product(pid))   # OK
# get_user(pid) -> mypy ERROR: ProductId is not UserId
print(f'type(uid): {type(uid)}')  # int (no-op at runtime)

# TYPE_CHECKING -- import only during mypy, not at runtime
if TYPE_CHECKING:
    from collections import OrderedDict  # avoids circular imports

# cast -- tell mypy the type (no-op at runtime!)
raw: Any = {'name': 'Alice'}
typed = cast(dict[str, str], raw)  # mypy trusts this
print(typed['name'])

# isinstance narrowing -- safe at runtime AND for mypy
def greet(name: Optional[str]) -> str:
    if name is None: return 'Hello, stranger!'
    return f'Hello, {name.upper()}'  # mypy: name is str here

print(greet(None))
print(greet('Alice'))

print()
print('mypy key concepts:')
print('  --strict          Enable all optional checks')
print('  reveal_type(x)    Print inferred type (runtime no-op)')
print('  cast(T, x)        Trust me, this is type T (runtime no-op)')
print('  TYPE_CHECKING     Import guard for type-only imports')

> **Interview Insight:** `cast()` is a lie to mypy with **no runtime effect**. If you cast incorrectly, you'll get runtime errors despite mypy passing. Prefer `isinstance` narrowing — it works both statically and at runtime.

## 6. Pydantic v2 — Runtime Validation

In [ ]:
try:
    from pydantic import BaseModel, Field, field_validator, model_validator, computed_field
    from pydantic import ValidationError
    from typing import Annotated
    import pydantic
    print(f'Pydantic {pydantic.VERSION}')

    class User(BaseModel):
        id: int
        name: str = Field(min_length=1, max_length=50)
        age: Annotated[int, Field(gt=0, lt=150)]
        email: str

        @field_validator('email')
        @classmethod
        def check_email(cls, v: str) -> str:
            if '@' not in v: raise ValueError('Invalid email')
            return v.lower()

        @field_validator('name')
        @classmethod
        def title_name(cls, v: str) -> str: return v.strip().title()

        @model_validator(mode='after')
        def adult_needs_email(self) -> 'User':
            if self.age >= 18 and not self.email:
                raise ValueError('Adults must have email')
            return self

        @computed_field
        @property
        def display(self) -> str: return f'{self.name} (id={self.id})'

    u = User(id=1, name='  alice ', age=30, email='ALICE@EXAMPLE.COM')
    print(f'name: {u.name}')     # Alice
    print(f'email: {u.email}')   # alice@example.com
    print(f'display: {u.display}')
    print(u.model_dump())

    json_str = u.model_dump_json()
    restored = User.model_validate_json(json_str)
    print(f'roundtrip: {restored.name}')

    try:
        User(id='x', name='', age=-1, email='bad')
    except ValidationError as e:
        for err in e.errors():
            print(f'  {err["loc"]}: {err["msg"]}')

except ImportError:
    print('pip install pydantic')
    print('v2 API: .dict() -> .model_dump(), .json() -> .model_dump_json()')

> **Interview Insight:** Pydantic v2 is 5-50x faster than v1 (Rust-based core). Key v2 migration: `.dict()` -> `.model_dump()`, `.json()` -> `.model_dump_json()`, `@validator` -> `@field_validator` with `@classmethod`.